## Phase 4  —  Hyperparameter Tuning

**What this notebook does:**
- Runs Bayesian Hyperparameter Optimization (HPO) across 20 jobs
- Saves the best model name 

## 1. Load config & session

In [1]:
import boto3
from sagemaker.core.helper.session_helper import Session
from sagemaker.core import image_uris
import json

with open("../data/config.json") as f:
    cfg = json.load(f)
 
ROLE = cfg["ROLE"]
BUCKET = cfg["BUCKET"]
PREFIX = cfg["PREFIX"]
REGION = cfg["REGION"]
SCALE_POS_WEIGHT  = cfg["SCALE_POS_WEIGHT"]

boto_session = boto3.Session(region_name=REGION)
sagemaker_session = Session(boto_session=boto_session)


TRAIN_S3 = f"s3://{BUCKET}/{PREFIX}/processed/train/"
VAL_S3  = f"s3://{BUCKET}/{PREFIX}/processed/test/"

print(f"Train data : {TRAIN_S3}")
print(f"Test data  : {VAL_S3}")
print(f"scale_pos_weight: {SCALE_POS_WEIGHT}")

[05/30/26 13:18:49] INFO     Found credentials in shared credentials file: ~/.aws/credentials   credentials.py:1392

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\ashri\AppData\Local\sagemaker\sagemaker\config.yaml


[05/30/26 13:18:50] INFO     Found credentials in shared credentials file: ~/.aws/credentials   credentials.py:1392

Train data : s3://credit-risk-mlops-svp/credit-risk/processed/train/
Test data  : s3://credit-risk-mlops-svp/credit-risk/processed/test/
scale_pos_weight: 11.4


## 2. Bayesian Hyperparameter Optimization

In [ ]:
from sagemaker.core import image_uris

xgb_image = image_uris.retrieve(
    framework="xgboost",
    region=REGION,
    version="1.7-1"
)

In [ ]:
import json, boto3, time, tarfile, os, tempfile

# ── Step 1: Package src/ into tar.gz with LF endings (fixes Windows CRLF) ────
temp_dir = tempfile.mkdtemp()
os.makedirs(temp_dir, exist_ok=True)

tar_path = os.path.join(temp_dir, "sourcecode.tar.gz")
lf_path = "../src/train.py"
lf_req_path = "../src/requirements.txt"


with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(lf_path,     arcname="train.py")
    tar.add(lf_req_path, arcname="requirements.txt")

print(f"Created tar.gz at {tar_path}")

# ── Step 2: Upload to S3 ──────────────────────────────────────────────────────
s3 = boto3.client("s3", region_name=REGION)
s3_key = f"{PREFIX}/source/sourcecode.tar.gz"
s3.upload_file(tar_path, BUCKET, s3_key)
source_s3_uri = f"s3://{BUCKET}/{s3_key}"
print(f"Source uploaded → {source_s3_uri}")

# ── Step 3: Static HPs ────────────────────────────────────────────────────────
# sagemaker_program + sagemaker_submit_directory switches container to script mode
static_hps = {
    "num_round":                  "300",
    "early_stopping_rounds":      "20",
    "scale_pos_weight":           str(SCALE_POS_WEIGHT),
    "sagemaker_program":          "train.py",       # ← tells container to run train.py
    "sagemaker_submit_directory": source_s3_uri     # ← tells container where to find it
}

# ── Step 4: Launch HPO ────────────────────────────────────────────────────────
sm_client = boto3.client("sagemaker", region_name=REGION)
tuning_job_name = f"credit-risk-tuning-{int(time.time())}"

print(f"\nStarting HPO job: {tuning_job_name}")
print("20 jobs, 4 parallel, Bayesian strategy...")

sm_client.create_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name,
    HyperParameterTuningJobConfig={
        "Strategy": "Bayesian",
        "ResourceLimits": {
            "MaxNumberOfTrainingJobs": 20,
            "MaxParallelTrainingJobs": 4
        },
        "HyperParameterTuningJobObjective": {
            "Type":       "Maximize",
            "MetricName": "validation:auc"
        },
        "ParameterRanges": {
            "ContinuousParameterRanges": [
                {"Name": "eta",              "MinValue": "0.01", "MaxValue": "0.3"},
                {"Name": "subsample",        "MinValue": "0.5",  "MaxValue": "1.0"},
                {"Name": "colsample_bytree", "MinValue": "0.5",  "MaxValue": "1.0"},
            ],
            "IntegerParameterRanges": [
                {"Name": "max_depth",        "MinValue": "3",  "MaxValue": "10"},
                {"Name": "min_child_weight", "MinValue": "1",  "MaxValue": "10"},
            ]
        }
    },
    TrainingJobDefinition={
        "StaticHyperParameters": static_hps,
        "AlgorithmSpecification": {
            "TrainingImage":     xgb_image,
            "TrainingInputMode": "File",
        },
        "RoleArn": ROLE,
        "InputDataConfig": [
            {
                "ChannelName": "train",
                "DataSource": {
                    "S3DataSource": {
                        "S3DataType":             "S3Prefix",
                        "S3Uri":                  TRAIN_S3,
                        "S3DataDistributionType": "FullyReplicated"
                    }
                }
            },
            {
                "ChannelName": "validation",
                "DataSource": {
                    "S3DataSource": {
                        "S3DataType":             "S3Prefix",
                        "S3Uri":                  VAL_S3,
                        "S3DataDistributionType": "FullyReplicated"
                    }
                }
            }
        ],
        "OutputDataConfig": {
            "S3OutputPath": f"s3://{BUCKET}/{PREFIX}/models/"
        },
        "ResourceConfig": {
            "InstanceType":   "ml.m5.xlarge",
            "InstanceCount":  1,
            "VolumeSizeInGB": 30
        },
        "StoppingCondition": {
            "MaxRuntimeInSeconds": 3600
        }
    }
)

# ── Step 5: Wait and track progress ──────────────────────────────────────────
while True:
    desc     = sm_client.describe_hyper_parameter_tuning_job(
        HyperParameterTuningJobName=tuning_job_name
    )
    status   = desc["HyperParameterTuningJobStatus"]
    counters = desc["TrainingJobStatusCounters"]
    completed = counters["Completed"]
    failed    = counters["NonRetryableError"]
    print(f"Status: {status} | Completed: {completed}/20 | Failed: {failed}")

    if status in ["Completed", "Failed", "Stopped"]:
        break
    time.sleep(60)

# ── Step 6: Results ───────────────────────────────────────────────────────────
if "BestTrainingJob" in desc:
    best     = desc["BestTrainingJob"]
    best_hps = best["TunedHyperParameters"]
    print(f"\nHPO Complete!")
    print(f"Best job : {best['TrainingJobName']}")
    print(f"Best AUC : {best['FinalHyperParameterTuningJobObjectiveMetric']['Value']:.4f}")
    print(f"\nBest Hyperparameters:")
    for k, v in best_hps.items():
        if k not in ("sagemaker_program", "sagemaker_submit_directory"):
            print(f"  {k:<20} : {v}")
    best_job_name = best["TrainingJobName"]
    print(f"\nSaved → best_job_name = {best_job_name}")
else:
    # Debug: print failure reason 
    print("\nHPO failed — checking first failed job...")
    failed_jobs = sm_client.list_training_jobs_for_hyper_parameter_tuning_job(
        HyperParameterTuningJobName=tuning_job_name,
        StatusEquals="Failed"
    )
    for job in failed_jobs["TrainingJobSummaries"][:2]:
        desc_job = sm_client.describe_training_job(TrainingJobName=job["TrainingJobName"])
        print(f"\nJob    : {job['TrainingJobName']}")
        print(f"Reason : {desc_job.get('FailureReason', 'N/A')}")

Created tar.gz at C:\Users\ashri\AppData\Local\Temp\tmppge2wp7o\sourcecode.tar.gz
Source uploaded → s3://credit-risk-mlops-svp/credit-risk/source/sourcecode.tar.gz

Starting HPO job: credit-risk-tuning-1780130848
20 jobs, 4 parallel, Bayesian strategy...
Status: InProgress | Completed: 0/20 | Failed: 0
Status: InProgress | Completed: 0/20 | Failed: 0
Status: InProgress | Completed: 0/20 | Failed: 0
Status: InProgress | Completed: 0/20 | Failed: 0
Status: InProgress | Completed: 2/20 | Failed: 0
Status: InProgress | Completed: 4/20 | Failed: 0
Status: InProgress | Completed: 5/20 | Failed: 0
Status: InProgress | Completed: 8/20 | Failed: 0
Status: InProgress | Completed: 8/20 | Failed: 0
Status: InProgress | Completed: 10/20 | Failed: 0
Status: InProgress | Completed: 11/20 | Failed: 0
Status: InProgress | Completed: 13/20 | Failed: 0
Status: InProgress | Completed: 16/20 | Failed: 0
Status: InProgress | Completed: 16/20 | Failed: 0
Status: InProgress | Completed: 19/20 | Failed: 0
Stat

In [15]:
import pandas as pd

# Show all tuning results sorted by AUC
tuning_job_name = desc["HyperParameterTuningJobName"]
response = sm_client.list_training_jobs_for_hyper_parameter_tuning_job(
    HyperParameterTuningJobName = tuning_job_name,
    SortBy = "FinalObjectiveMetricValue",
    SortOrder = "Descending"
)

results = []
for job in response["TrainingJobSummaries"]:
    results.append({
        "job_name": job["TrainingJobName"],
        "auc": job.get("FinalHyperParameterTuningJobObjectiveMetric", {}).get("Value", 0),
        "status": job["TrainingJobStatus"]
    })

results_df = pd.DataFrame(results)
print(f"Top 10 tuning results:")
display(results_df.head(10))

Top 10 tuning results:


,job_name,auc,status
0,credit-risk-tuning-1780130848-020-2dfb5b06,0.76697,Completed
1,credit-risk-tuning-1780130848-017-26771e9e,0.76591,Completed
2,credit-risk-tuning-1780130848-011-3bb6ae6e,0.76569,Completed
3,credit-risk-tuning-1780130848-010-eb2d34a2,0.76560,Completed
4,credit-risk-tuning-1780130848-015-67b48191,0.76546,Completed
5,credit-risk-tuning-1780130848-016-229b56f5,0.76542,Completed
6,credit-risk-tuning-1780130848-018-a69ba12e,0.76537,Completed
7,credit-risk-tuning-1780130848-019-e9d4f9eb,0.76448,Completed
8,credit-risk-tuning-1780130848-004-eaccce3f,0.76314,Completed
9,credit-risk-tuning-1780130848-008-6378350c,0.76308,Completed


In [ ]:
# ── Get best job name and hyperparameters ─────────────────────────────────────
desc     = sm_client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name
)
best     = desc["BestTrainingJob"]
best_job_name = best["TrainingJobName"]
best_auc = best['FinalHyperParameterTuningJobObjectiveMetric']['Value']

# ── Get best model artifact (equivalent to best_estimator.model_data) ─────────
best_job_desc  = sm_client.describe_training_job(TrainingJobName=best_job_name)
best_model_data = best_job_desc["ModelArtifacts"]["S3ModelArtifacts"]

# ── Best hyperparameters ───────────────────────────────────────────────────────
best_hps = best["TunedHyperParameters"]

# Save to config 
with open("../data/config.json") as f:
    cfg = json.load(f)

cfg["BEST_HPO_JOB_NAME"] = best_job_name
cfg["BEST_MODEL_DATA"]   = best_model_data
cfg["TUNING_JOB_NAME"]   = tuning_job_name
cfg["BEST_AUC"]          = round(best_auc, 4)
cfg["BEST_HPS"]          = best_hps

with open("../data/config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print("Config updated.")

Config updated.
